# Stage 6 & 7: Algorithm 3 — LightGBM Gradient Boosting

**Owner:** Member C  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/lightgbm_tuned.pkl`  

## Scope
LightGBM is a **gradient boosting** algorithm that builds trees sequentially,
where each new tree corrects the errors of the previous one. It uses leaf-wise tree growth
instead of level-wise, making it faster and more accurate than traditional GBMs.

Key advantages:
- Leaf-wise tree growth: finds the best split globally, not level by level
- `scale_pos_weight`: directly handles class imbalance by upweighting the minority (late) class
- Handles large datasets efficiently with histogram-based splitting

This notebook:
1. Loads the final modeling datasets from `data/final/`
2. Trains a default LightGBM baseline
3. Computes `scale_pos_weight` from the training class ratio
4. Performs 5-Fold Stratified RandomizedSearchCV tuning on `X_train` only
5. Evaluates the tuned model on `X_val` with full metrics
6. Reports feature importances and exports the tuned model artifact


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')
import lightgbm as lgb

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)

FINAL_DIR  = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

neg_count = len(y_train) - y_train.sum()
pos_count = y_train.sum()
scale_pos_weight_val = neg_count / pos_count

print(f'X_train : {X_train.shape} | Late rate: {y_train.mean()*100:.2f}%')
print(f'X_val   : {X_val.shape}   | Late rate: {y_val.mean()*100:.2f}%')
print(f'scale_pos_weight (on-time/late ratio): {scale_pos_weight_val:.2f}')


X_train : (67529, 67) | Late rate: 9.03%
X_val   : (14470, 67)   | Late rate: 5.34%
scale_pos_weight (on-time/late ratio): 10.08


## 2. Default LightGBM Baseline
An untuned LightGBM establishes the baseline before any hyperparameter search.

In [2]:
lgb_base = lgb.LGBMClassifier(random_state=42, verbose=-1)
lgb_base.fit(X_train, y_train)
base_probs = lgb_base.predict_proba(X_val)[:, 1]

print('=== DEFAULT LIGHTGBM BASELINE ===')
print(f'Val ROC-AUC : {roc_auc_score(y_val, base_probs):.4f}')
print(f'Val PR-AUC  : {average_precision_score(y_val, base_probs):.4f}')


=== DEFAULT LIGHTGBM BASELINE ===
Val ROC-AUC : 0.7438
Val PR-AUC  : 0.1439


## 3. Hyperparameter Tuning — 5-Fold Stratified RandomizedSearchCV
**Parameters tuned:**
- `n_estimators`: Number of boosting rounds (trees).
- `learning_rate`: Shrinkage factor per tree. Lower = better generalization but slower.
- `max_depth` & `num_leaves`: Controls tree complexity. `num_leaves` is specific to LightGBM.
- `subsample` & `colsample_bytree`: Row/column subsampling per tree (reduces overfitting).
- `scale_pos_weight`: Weights the minority class (late orders) to handle 10:1 imbalance.


In [3]:
param_dist = {
    'n_estimators': [150, 300, 450],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [4, 6, 8, 10],
    'num_leaves': [15, 31, 63],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'scale_pos_weight': [1.0, scale_pos_weight_val / 2.0, scale_pos_weight_val]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rand = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=42, verbose=-1),
    param_dist, n_iter=12, scoring='average_precision',
    cv=cv, random_state=42, n_jobs=-1, verbose=1
)
rand.fit(X_train, y_train)

print(f'Best Parameters : {rand.best_params_}')
print(f'Best CV PR-AUC  : {rand.best_score_:.4f}')
best_lgb = rand.best_estimator_


Fitting 5 folds for each of 12 candidates, totalling 60 fits


Best Parameters : {'subsample': 1.0, 'scale_pos_weight': np.float64(5.03879593175853), 'num_leaves': 63, 'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
Best CV PR-AUC  : 0.3203


## 4. Validation Set Evaluation

In [4]:
val_probs = best_lgb.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

print('=== TUNED LIGHTGBM — VALIDATION SET METRICS ===')
print(f'Val ROC-AUC        : {roc_auc_score(y_val, val_probs):.4f}')
print(f'Val PR-AUC         : {average_precision_score(y_val, val_probs):.4f}')
print(f'Val F1-Score (0.5) : {f1_score(y_val, val_preds):.4f}')
print(f'Val Recall   (0.5) : {recall_score(y_val, val_preds):.4f}')
print(f'Val Precision(0.5) : {precision_score(y_val, val_preds):.4f}')

print('\nClassification Report:')
print(classification_report(y_val, val_preds, target_names=['On-Time (0)', 'Late (1)']))

cm = confusion_matrix(y_val, val_preds)
print(f'Confusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')


=== TUNED LIGHTGBM — VALIDATION SET METRICS ===
Val ROC-AUC        : 0.7408
Val PR-AUC         : 0.1508
Val F1-Score (0.5) : 0.1558
Val Recall   (0.5) : 0.1138
Val Precision(0.5) : 0.2465

Classification Report:
              precision    recall  f1-score   support

 On-Time (0)       0.95      0.98      0.97     13697
    Late (1)       0.25      0.11      0.16       773

    accuracy                           0.93     14470
   macro avg       0.60      0.55      0.56     14470
weighted avg       0.91      0.93      0.92     14470

Confusion Matrix:
  TN=13,428  FP=269
  FN=685  TP=88


## 5. Feature Importance (Split Count)
LightGBM feature importance measures how many times each feature was used in tree splits.

In [5]:
feat_imp = pd.Series(best_lgb.feature_importances_, index=X_train.columns)
top10 = feat_imp.sort_values(ascending=False).head(10)

print('Top 10 Most Important Features (LightGBM Split Count):')
for feat, imp in top10.items():
    print(f'  {feat:<45} : {imp}')


Top 10 Most Important Features (LightGBM Split Count):
  estimated_delivery_days                       : 1247
  customer_longitude                            : 1211
  customer_latitude                             : 1137
  shipping_limit_lead_days                      : 1072
  approval_lag_hours                            : 955
  dominant_product_category_target_enc          : 891
  freight_to_price_ratio                        : 779
  customer_state_target_enc                     : 743
  volume_per_item                               : 730
  total_volume_cm3                              : 714


## 6. Save Tuned Model Artifact

In [6]:
joblib.dump(best_lgb, MODELS_DIR / 'lightgbm_tuned.pkl')
print('Saved: models/lightgbm_tuned.pkl')
print(f'n_estimators     : {best_lgb.n_estimators}')
print(f'learning_rate    : {best_lgb.learning_rate}')
print(f'num_leaves       : {best_lgb.num_leaves}')
print(f'scale_pos_weight : {best_lgb.scale_pos_weight}')


Saved: models/lightgbm_tuned.pkl


n_estimators     : 300
learning_rate    : 0.05
num_leaves       : 63
scale_pos_weight : 5.03879593175853
